# CS3807 Deep Learning Laboratory
## Experiment 4: Comparative Study of Deep CNN Architectures Using Transfer Learning

**Name:** Kalpana Bhaskar  
**Roll No:** 24011101049  
**Branch:** B.Tech AI & DS (Section A)  
**Semester:** V  
**AY:** 2026–27  

---
## Task 1: Dataset Preparation
Load CIFAR-10, normalize pixel values, display sample images, and print dataset dimensions.

In [ ]:
# Standard imports
import numpy as np
import matplotlib.pyplot as plt
import time

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.applications import VGG16, ResNet50, MobileNetV2
from tensorflow.keras.utils import to_categorical

from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

print("TensorFlow version:", tf.__version__)

In [ ]:
# CIFAR-10 class names for reference
CLASS_NAMES = ['Airplane', 'Automobile', 'Bird', 'Cat', 'Deer',
               'Dog', 'Frog', 'Horse', 'Ship', 'Truck']

# Load CIFAR-10 dataset (auto-downloads if not cached)
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

# Normalize pixel values from [0, 255] to [0, 1]
x_train = x_train.astype('float32') / 255.0
x_test  = x_test.astype('float32')  / 255.0

# One-hot encode the labels (10 classes)
y_train_cat = to_categorical(y_train, 10)
y_test_cat  = to_categorical(y_test,  10)

print("Training images shape :", x_train.shape)   # (50000, 32, 32, 3)
print("Testing  images shape :", x_test.shape)    # (10000, 32, 32, 3)
print("Training labels shape :", y_train_cat.shape)
print("Testing  labels shape :", y_test_cat.shape)

In [ ]:
# ── Plot 1: Sample CIFAR-10 Images ──────────────────────────────────────────
# Show one random image per class in a single row
fig, axes = plt.subplots(1, 10, figsize=(15, 2))
fig.suptitle('Sample CIFAR-10 Images (one per class)', fontsize=12)

for cls_idx in range(10):
    # Pick first image belonging to this class
    idx = np.where(y_train.flatten() == cls_idx)[0][0]
    axes[cls_idx].imshow(x_train[idx])
    axes[cls_idx].set_title(CLASS_NAMES[cls_idx], fontsize=8)
    axes[cls_idx].axis('off')

plt.tight_layout()
plt.savefig('plot1_cifar10_samples.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: plot1_cifar10_samples.png")

---
## Task 2: Transfer Learning — Load Pretrained Model & Build Classifier
We use **VGG16** pretrained on ImageNet.  
Steps: load weights → freeze conv base → add custom head → compile.

In [ ]:
# VGG16 expects at least 32×32 images; we resize CIFAR-10 to 48×48
# to give the network a little more spatial information.
IMG_SIZE = 48

def resize_dataset(images, size):
    """Resize a batch of images using tf.image."""
    return tf.image.resize(images, [size, size]).numpy()

print("Resizing images to", IMG_SIZE, "×", IMG_SIZE, "...")
x_train_rs = resize_dataset(x_train, IMG_SIZE)
x_test_rs  = resize_dataset(x_test,  IMG_SIZE)
print("Done. Train shape:", x_train_rs.shape)

In [ ]:
# ── Build Transfer Learning Model (VGG16 base) ───────────────────────────────

# Step 1: Load VGG16 without its top classification layers
base_model = VGG16(
    weights='imagenet',          # pretrained ImageNet weights
    include_top=False,           # remove the original Dense classifier
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

# Step 2: Freeze all convolutional layers — we won't train them yet
base_model.trainable = False

# Step 3: Add our custom classification head
model = models.Sequential([
    base_model,                                        # frozen VGG16 feature extractor
    layers.GlobalAveragePooling2D(),                   # reduce spatial dims to 1-D vector
    layers.Dense(256, activation='relu'),              # fully-connected hidden layer
    layers.Dropout(0.5),                               # regularization to reduce overfitting
    layers.Dense(10, activation='softmax')             # 10-class output
])

# Step 4: Compile with Adam optimizer and categorical cross-entropy loss
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

---
## Task 3: Model Training (Frozen Base)
Train only the custom Dense head while the VGG16 conv layers stay frozen.

In [ ]:
# Training parameters as specified in the lab sheet
BATCH_SIZE = 32
EPOCHS_PHASE1 = 10   # initial training with frozen base

start_time = time.time()

history_phase1 = model.fit(
    x_train_rs, y_train_cat,
    validation_data=(x_test_rs, y_test_cat),
    batch_size=BATCH_SIZE,
    epochs=EPOCHS_PHASE1,
    verbose=1
)

phase1_time = time.time() - start_time
print(f"\nPhase 1 training time: {phase1_time:.1f} seconds")

In [ ]:
# Helper function to plot accuracy and loss curves
def plot_history(history, title_suffix='', save_prefix='plot'):
    epochs = range(1, len(history.history['accuracy']) + 1)

    # ── Plot 2 & 5 variant: Training & Validation Accuracy
    plt.figure(figsize=(6, 4))
    plt.plot(epochs, history.history['accuracy'],     label='Train Accuracy')
    plt.plot(epochs, history.history['val_accuracy'], label='Val Accuracy', linestyle='--')
    plt.title(f'Training & Validation Accuracy {title_suffix}')
    plt.xlabel('Epoch'); plt.ylabel('Accuracy')
    plt.legend(); plt.tight_layout()
    plt.savefig(f'{save_prefix}_accuracy.png', dpi=150, bbox_inches='tight')
    plt.show()

    # ── Plot 4 & 5 variant: Training & Validation Loss
    plt.figure(figsize=(6, 4))
    plt.plot(epochs, history.history['loss'],     label='Train Loss')
    plt.plot(epochs, history.history['val_loss'], label='Val Loss', linestyle='--')
    plt.title(f'Training & Validation Loss {title_suffix}')
    plt.xlabel('Epoch'); plt.ylabel('Loss')
    plt.legend(); plt.tight_layout()
    plt.savefig(f'{save_prefix}_loss.png', dpi=150, bbox_inches='tight')
    plt.show()

# Plot Phase 1 curves
plot_history(history_phase1, title_suffix='(Phase 1 – Frozen Base)', save_prefix='plot2_phase1')
print("Saved: plot2_phase1_accuracy.png, plot2_phase1_loss.png")

---
## Task 4: Fine Tuning
Unfreeze the last convolutional block of VGG16 and train for a few more epochs with a lower learning rate.

In [ ]:
# Unfreeze only the last convolutional block of VGG16 (block5)
# Earlier blocks retain their ImageNet features — changing them would be wasteful.
base_model.trainable = True

# Freeze every layer EXCEPT the last block (block5_conv1, block5_conv2, block5_conv3)
for layer in base_model.layers:
    if 'block5' not in layer.name:
        layer.trainable = False

# Count how many layers are now trainable
trainable_count = sum(1 for l in model.layers if l.trainable)
print(f"Trainable layers after unfreezing block5: {trainable_count}")

# Recompile with a much smaller learning rate to avoid destroying pretrained weights
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

EPOCHS_PHASE2 = 5   # fine-tuning needs fewer epochs

start_time = time.time()

history_phase2 = model.fit(
    x_train_rs, y_train_cat,
    validation_data=(x_test_rs, y_test_cat),
    batch_size=BATCH_SIZE,
    epochs=EPOCHS_PHASE2,
    verbose=1
)

phase2_time = time.time() - start_time
print(f"\nPhase 2 fine-tuning time: {phase2_time:.1f} seconds")

In [ ]:
# Plot fine-tuning curves
plot_history(history_phase2, title_suffix='(Phase 2 – Fine Tuning)', save_prefix='plot3_phase2')
print("Saved: plot3_phase2_accuracy.png, plot3_phase2_loss.png")

In [ ]:
# ── Combine both phases into single accuracy / loss curves for the report ──
# Merge history dictionaries
combined = {}
for key in history_phase1.history:
    combined[key] = history_phase1.history[key] + history_phase2.history[key]

total_epochs = EPOCHS_PHASE1 + EPOCHS_PHASE2
epochs_range = range(1, total_epochs + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Training & Validation Accuracy (combined)
ax1.plot(epochs_range, combined['accuracy'],     label='Train')
ax1.plot(epochs_range, combined['val_accuracy'], label='Validation', linestyle='--')
ax1.axvline(EPOCHS_PHASE1, color='gray', linestyle=':', label='Fine-tuning starts')
ax1.set_title('Accuracy – All Epochs'); ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy')
ax1.legend()

# Training & Validation Loss (combined)
ax2.plot(epochs_range, combined['loss'],     label='Train')
ax2.plot(epochs_range, combined['val_loss'], label='Validation', linestyle='--')
ax2.axvline(EPOCHS_PHASE1, color='gray', linestyle=':', label='Fine-tuning starts')
ax2.set_title('Loss – All Epochs'); ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
ax2.legend()

plt.tight_layout()
plt.savefig('plot4_combined_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: plot4_combined_curves.png")

---
## Task 5: Model Evaluation
Evaluate using accuracy, precision, recall, F1-score, confusion matrix, and classification report.

In [ ]:
# Evaluate on the test set
test_loss, test_acc = model.evaluate(x_test_rs, y_test_cat, verbose=0)
print(f"Test Loss    : {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}  ({test_acc*100:.2f}%)")

# Training accuracy from last epoch of Phase 2
train_acc = history_phase2.history['accuracy'][-1]
print(f"Train Accuracy (last epoch): {train_acc:.4f}  ({train_acc*100:.2f}%)")

In [ ]:
# Get predictions on the test set
y_pred_prob = model.predict(x_test_rs, verbose=0)   # shape: (10000, 10)
y_pred      = np.argmax(y_pred_prob, axis=1)        # predicted class indices
y_true      = y_test.flatten()                      # true class indices

In [ ]:
# ── Classification Report (Precision, Recall, F1) ───────────────────────────
report = classification_report(y_true, y_pred, target_names=CLASS_NAMES)
print("Classification Report:")
print(report)

In [ ]:
# ── Plot 6: Confusion Matrix ─────────────────────────────────────────────────
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm,
    annot=True, fmt='d', cmap='Blues',
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES
)
plt.title('Confusion Matrix – VGG16 Transfer Learning (CIFAR-10)')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('plot5_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: plot5_confusion_matrix.png")

In [ ]:
# ── Plot 7 (Optional): Misclassified Images ──────────────────────────────────
# Find indices where prediction != true label
wrong_idx = np.where(y_pred != y_true)[0]

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
fig.suptitle('Misclassified Images (True → Predicted)', fontsize=12)

for i, ax in enumerate(axes.flat):
    if i >= len(wrong_idx):
        break
    idx = wrong_idx[i]
    ax.imshow(x_test[idx])   # show original 32×32 image
    ax.set_title(f"{CLASS_NAMES[y_true[idx]]}\n→ {CLASS_NAMES[y_pred[idx]]}", fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.savefig('plot6_misclassified.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: plot6_misclassified.png")

---
## Additional Exercises
### AE-1: Compare Accuracy Before and After Fine Tuning

In [ ]:
# Best validation accuracy from each phase
acc_before = max(history_phase1.history['val_accuracy'])
acc_after  = max(history_phase2.history['val_accuracy'])

print("Before fine-tuning (best val acc):", f"{acc_before*100:.2f}%")
print("After  fine-tuning (best val acc):", f"{acc_after*100:.2f}%")
print("Improvement :", f"{(acc_after - acc_before)*100:.2f}%")

# Bar chart comparison
plt.figure(figsize=(5, 4))
plt.bar(['Before Fine-Tuning', 'After Fine-Tuning'],
        [acc_before, acc_after], color=['steelblue', 'darkorange'])
plt.ylabel('Validation Accuracy')
plt.title('Effect of Fine-Tuning on Accuracy')
plt.ylim(0, 1)
for i, v in enumerate([acc_before, acc_after]):
    plt.text(i, v + 0.01, f'{v*100:.2f}%', ha='center', fontsize=11)
plt.tight_layout()
plt.savefig('plot7_finetuning_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

### AE-2: Hyperparameter Study — Compare Adam vs SGD Optimizer

In [ ]:
# Build a fresh model identical to Phase 1 but with SGD to compare
def build_vgg16_model(optimizer, lr, input_shape=(IMG_SIZE, IMG_SIZE, 3)):
    """Build frozen-base VGG16 model with a given optimizer and learning rate."""
    base = VGG16(weights='imagenet', include_top=False, input_shape=input_shape)
    base.trainable = False   # freeze all conv layers

    m = models.Sequential([
        base,
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(10, activation='softmax')
    ])

    if optimizer == 'adam':
        opt = keras.optimizers.Adam(learning_rate=lr)
    else:
        opt = keras.optimizers.SGD(learning_rate=lr, momentum=0.9)

    m.compile(optimizer=opt, loss='categorical_crossentropy', metrics=['accuracy'])
    return m

# Train SGD model for 10 epochs (quick study)
print("Training with SGD optimizer...")
model_sgd = build_vgg16_model('sgd', lr=0.01)
hist_sgd  = model_sgd.fit(
    x_train_rs, y_train_cat,
    validation_data=(x_test_rs, y_test_cat),
    batch_size=32, epochs=10, verbose=0
)
print("SGD done. Best val acc:", f"{max(hist_sgd.history['val_accuracy'])*100:.2f}%")

In [ ]:
# Compare Adam (Phase 1) vs SGD on the same plot
plt.figure(figsize=(7, 4))
plt.plot(history_phase1.history['val_accuracy'], label='Adam (lr=0.001)')
plt.plot(hist_sgd.history['val_accuracy'],       label='SGD  (lr=0.01)', linestyle='--')
plt.title('Adam vs SGD – Validation Accuracy')
plt.xlabel('Epoch'); plt.ylabel('Val Accuracy')
plt.legend(); plt.tight_layout()
plt.savefig('plot8_adam_vs_sgd.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: plot8_adam_vs_sgd.png")

### AE-3: Hyperparameter Study — Compare Batch Sizes (16, 32, 64)

In [ ]:
batch_results = {}

for bs in [16, 32, 64]:
    print(f"Training with batch_size={bs} ...")
    m = build_vgg16_model('adam', lr=0.001)
    h = m.fit(
        x_train_rs, y_train_cat,
        validation_data=(x_test_rs, y_test_cat),
        batch_size=bs, epochs=10, verbose=0
    )
    batch_results[bs] = h.history['val_accuracy']
    print(f"  Best val acc: {max(h.history['val_accuracy'])*100:.2f}%")

# Plot
plt.figure(figsize=(7, 4))
for bs, accs in batch_results.items():
    plt.plot(accs, label=f'Batch {bs}')
plt.title('Batch Size Study – Validation Accuracy')
plt.xlabel('Epoch'); plt.ylabel('Val Accuracy')
plt.legend(); plt.tight_layout()
plt.savefig('plot9_batch_size_study.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: plot9_batch_size_study.png")

### AE-4: Architecture Parameter Comparison Table

In [ ]:
# Static comparison table from the lab sheet — fill Accuracy from your results
import pandas as pd

comparison_data = {
    'Architecture': ['LeNet-5', 'AlexNet', 'VGG16', 'GoogleNet', 'ResNet50'],
    'Depth'       : [7,         8,         16,      22,          50        ],
    'Parameters'  : ['60K',   '61M',     '138M',   '6.8M',    '25.6M'    ],
    'Main Innovation': [
        'First CNN',
        'ReLU + Dropout',
        'Deep 3×3 filters',
        'Inception Modules',
        'Residual Learning'
    ],
    'Accuracy (CIFAR-10 %)': [
        'Fill from results',
        'Fill from results',
        f'{test_acc*100:.2f}',   # from our experiment
        'Fill from results',
        'Fill from results'
    ]
}

df = pd.DataFrame(comparison_data)
print(df.to_string(index=False))

---
## Results Summary

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

precision = precision_score(y_true, y_pred, average='weighted')
recall    = recall_score(y_true, y_pred, average='weighted')
f1        = f1_score(y_true, y_pred, average='weighted')
total_params = model.count_params()
total_time   = phase1_time + phase2_time

print("=" * 40)
print("         FINAL RESULTS SUMMARY")
print("=" * 40)
print(f"Training Accuracy : {train_acc*100:.2f}%")
print(f"Testing  Accuracy : {test_acc*100:.2f}%")
print(f"Precision         : {precision:.4f}")
print(f"Recall            : {recall:.4f}")
print(f"F1-Score          : {f1:.4f}")
print(f"Training Time     : {total_time:.1f} seconds")
print(f"Total Parameters  : {total_params:,}")
print("=" * 40)

---
## Discussion Questions (Answers)

**Q1. Why is AlexNet a breakthrough?**  
AlexNet (2012) won ILSVRC by a large margin using ReLU activations (faster training), Dropout (reduced overfitting), and GPU training — proving deep CNNs could scale to real-world vision tasks.

**Q2. Why does VGG16 use only 3×3 filters?**  
Two stacked 3×3 conv layers have the same receptive field as one 5×5 layer but use fewer parameters and introduce more non-linearities, improving both efficiency and accuracy.

**Q3. Advantages of the Inception module?**  
It applies 1×1, 3×3, 5×5 convolutions and max-pooling in parallel, capturing features at multiple scales. 1×1 convolutions reduce depth before larger filters, cutting parameters drastically.

**Q4. Purpose of residual learning?**  
Skip connections let the network learn F(x) = H(x) − x instead of H(x) directly. This avoids the vanishing gradient problem in very deep networks, enabling stable training of 50–152 layer models.

**Q5. Differentiate LeNet and ResNet.**  
LeNet-5 (7 layers, 60K params) was designed for 32×32 greyscale digits. ResNet-50 (50 layers, 25.6M params) uses residual blocks to solve vanishing gradients and achieves state-of-the-art on ImageNet-scale tasks.

**Q6. What is Transfer Learning?**  
Reusing a model pre-trained on a large dataset (ImageNet) for a different but related task. The pretrained features are adapted by replacing and training only the final classification layers.

**Q7. Why is fine-tuning required?**  
Freezing all layers is fast but leaves domain-specific patterns unlearned. Unfreezing the last conv block lets the model adapt higher-level features to the target domain (CIFAR-10), improving accuracy.

**Q8. Dilated vs Transpose Convolution.**  
Dilated convolution expands the receptive field by inserting gaps between kernel elements without extra parameters — used in segmentation. Transpose convolution learns to upsample (increase spatial size) — used in generators and decoders.

**Q9. Why do pretrained models converge faster?**  
Pretrained weights are already good feature detectors (edges, textures, shapes). Only the task-specific head needs to be learned, so convergence requires far fewer gradient updates.

**Q10. Computational complexity: LeNet vs ResNet.**  
LeNet-5 has ~60K parameters and ~0.3M FLOPs — trivial to run on CPU. ResNet-50 has 25.6M parameters and ~4G FLOPs — requires GPU for practical training. ResNet trades compute for much higher accuracy on complex datasets.